In [1]:
import emat
import pandas as pd
import pickle

emat.require_version('0.5.1')

emat 0.6.0, plotly 6.1.2


### Remote I/O

In [2]:
pickle_filename = './data/processed/emat_metamodel.pkl'
output_filename = "./data/processed/meta-model-regression-estimation-results.csv"

### Data Reads

In [3]:
with open(pickle_filename, 'rb') as file:
    metamodel = pickle.load(file)

### Make Dataframes

In [ ]:
column_names = ['coeff_00', 
                'coeff_01', 
                'coeff_02', 
                'coeff_03', 
                'coeff_04', 
                'coeff_05', 
                'coeff_06', 
                'coeff_07', 
                'coeff_08', 
                'coeff_09', 
                'coeff_10', 
                'coeff_11', 
                'coeff_12', 
                'coeff_13', 
                'coeff_14'
]

In [5]:
t_stats_df = pd.DataFrame(metamodel.function.regression.estimators_[0].t_, columns = column_names)
t_stats_df["measure"] = "t-statistic"

In [6]:
coeff_df = pd.DataFrame(metamodel.function.regression.estimators_[0].coef_, columns = column_names)
coeff_df["measure"] = "coefficients"

In [7]:
intercept_df = pd.DataFrame(metamodel.function.regression.estimators_[0].intercept_, columns = ["intercept"])
coeff_df = pd.concat((coeff_df, intercept_df), axis = 1)

In [8]:
r_squared_df = pd.DataFrame(metamodel.function.regression.estimators_[0].r2, columns = ["r-squared"])
r_squared_df.reset_index(drop = False, inplace = True)
r_squared_df

,index,r-squared
0,Auto Mode Share,0.973897
1,Bike Mode Share,0.962534
2,Transit Mode Share,0.963846
3,Walk Mode Share,0.902791
4,Transit PMT,0.917603
5,Regional VMT,0.669360
6,Regional VMT per Capita,0.565693
7,Regional VHT,0.703464
8,Regional VHTper Capita,0.514490
9,Average Commute Time by Auto,0.743420


In [9]:
temp_df = pd.concat((coeff_df, r_squared_df), axis=1).rename(columns={'index': 'model_name'})
long_k_df = temp_df.melt(id_vars=['model_name', 'measure', 'r-squared'], var_name='coeff_name', value_name='coeff_value')
long_k_df = long_k_df.sort_values(["model_name", "measure", "coeff_name"]).reset_index(drop = True)
long_k_df.head()

,model_name,measure,r-squared,coeff_name,coeff_value
0,AM Travel Time Index,coefficients,0.876879,coeff_00,0.107444
1,AM Travel Time Index,coefficients,0.876879,coeff_01,0.007704
2,AM Travel Time Index,coefficients,0.876879,coeff_02,-0.242535
3,AM Travel Time Index,coefficients,0.876879,coeff_03,0.034419
4,AM Travel Time Index,coefficients,0.876879,coeff_04,-0.135380


In [10]:
temp_df = pd.concat((t_stats_df, r_squared_df), axis=1).rename(columns={'index': 'model_name'})
long_t_df = temp_df.melt(id_vars=['model_name', 'measure', 'r-squared'], var_name='coeff_name', value_name='t_stat_value')
long_t_df.drop(["r-squared", "measure"], axis = 1, inplace = True)
long_t_df = long_t_df.sort_values(["model_name", "coeff_name"]).reset_index(drop = True)
long_t_df.head()

,model_name,coeff_name,t_stat_value
0,AM Travel Time Index,coeff_00,9.878630
1,AM Travel Time Index,coeff_01,0.718902
2,AM Travel Time Index,coeff_02,-22.695643
3,AM Travel Time Index,coeff_03,3.100604
4,AM Travel Time Index,coeff_04,-3.892039


In [11]:
summary_df = pd.merge(long_k_df, long_t_df, how = "left", on = ["model_name", "coeff_name"])
summary_df["model_id"] =pd.factorize(summary_df["model_name"])[0] + 1
summary_df = summary_df[["model_id", "model_name", "r-squared", "coeff_name", "coeff_value", "t_stat_value"]]    
summary_df.head()

,model_id,model_name,r-squared,coeff_name,coeff_value,t_stat_value
0,1,AM Travel Time Index,0.876879,coeff_00,0.107444,9.878630
1,1,AM Travel Time Index,0.876879,coeff_01,0.007704,0.718902
2,1,AM Travel Time Index,0.876879,coeff_02,-0.242535,-22.695643
3,1,AM Travel Time Index,0.876879,coeff_03,0.034419,3.100604
4,1,AM Travel Time Index,0.876879,coeff_04,-0.135380,-3.892039


In [12]:
summary_df.to_csv(output_filename, index=False)